In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_6.py ──
"""
Shared infrastructure for Exercise 6 — Graph Neural Networks.

Contains: Cora dataset loading (with Karate Club fallback), graph
normalisation, train/val/test split, kailash-ml engine setup,
graph visualisation helpers, and training harness.

Technique-specific code (GCN, GAT, GraphSAGE layers) does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex6_gnns"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cora"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# GRAPH DATA LOADING
# ════════════════════════════════════════════════════════════════════════


def load_cora() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, int]:
    """Cora — 2708 papers, 1433 bag-of-words features, 7 classes.

    Returns:
        X_np: node features (N, F)
        A_np: dense adjacency matrix (N, N)
        y_np: node labels (N,)
        edge_index_np: edge list (2, E) for link prediction
        dataset_name: "Cora"
        n_classes: 7
    """
    from torch_geometric.datasets import Planetoid

    dataset = Planetoid(root=str(DATA_DIR), name="Cora")
    # torch_geometric.data.Data has dynamic attributes (num_nodes/x/y/edge_index);
    # use Any to avoid type-checker false positives without losing runtime fidelity.
    data: Any = dataset[0]
    n = data.num_nodes
    X_np = data.x.numpy().astype(np.float32)
    y_np = data.y.numpy().astype(np.int64)

    # Build a dense adjacency matrix from the edge_index. Cora has ~10k
    # directed edges (5278 undirected) over 2708 nodes; the dense matrix
    # is ~7M entries which fits comfortably in CPU memory.
    A_np = np.zeros((n, n), dtype=np.float32)
    edge_index_np = data.edge_index.numpy()
    src = edge_index_np[0]
    dst = edge_index_np[1]
    A_np[src, dst] = 1.0
    A_np[dst, src] = 1.0  # symmetrise just in case
    n_classes = int(dataset.num_classes)
    return X_np, A_np, y_np, edge_index_np, "Cora", n_classes


def load_karate() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, int]:
    """Zachary's Karate Club — 34 nodes, 78 edges, 2 factions."""
    import networkx as nx

    G = nx.karate_club_graph()
    n = G.number_of_nodes()
    A_np = nx.to_numpy_array(G, dtype=np.float32)
    labels = np.array(
        [0 if G.nodes[i]["club"] == "Mr. Hi" else 1 for i in range(n)],
        dtype=np.int64,
    )
    # Karate has no node features; use one-hot identity (transductive)
    X_np = np.eye(n, dtype=np.float32)
    # Build edge_index from adjacency
    src, dst = np.where(A_np > 0)
    edge_index_np = np.stack([src, dst]).astype(np.int64)
    return X_np, A_np, labels, edge_index_np, "Karate Club", 2


def load_graph_data() -> dict:
    """Load Cora (with Karate fallback) and return all graph tensors.

    Returns a dict with keys:
        X, A, y, A_norm, A_hat — torch tensors on device
        X_np, A_np, y_np, edge_index_np — numpy arrays
        train_mask, val_mask, test_mask — boolean masks on device
        N, F_dim, n_classes, n_edges_undirected, dataset_name — scalars
    """
    try:
        X_np, A_np, y_np, edge_index_np, dataset_name, n_classes = load_cora()
    except Exception as exc:
        print(
            f"Could not load Cora ({type(exc).__name__}: {exc}); "
            "falling back to Karate Club"
        )
        X_np, A_np, y_np, edge_index_np, dataset_name, n_classes = load_karate()

    N = X_np.shape[0]
    F_dim = X_np.shape[1]
    n_edges_undirected = int(A_np.sum() // 2)
    print(
        f"Graph: {dataset_name} — {N} nodes, {n_edges_undirected} undirected edges, "
        f"feature_dim={F_dim}, classes={n_classes}"
    )
    class_counts = ", ".join(f"c{c}={int((y_np == c).sum())}" for c in range(n_classes))
    print(f"  per-class counts: {class_counts}")

    X = torch.from_numpy(X_np).to(device)
    A = torch.from_numpy(A_np).to(device)
    y = torch.from_numpy(y_np).to(device)

    # Add self-loops and build the symmetric Laplacian D^{-1/2} A D^{-1/2}
    A_hat = A + torch.eye(N, device=device)
    deg = A_hat.sum(dim=1)
    d_inv_sqrt = deg.pow(-0.5)
    d_inv_sqrt[torch.isinf(d_inv_sqrt)] = 0.0
    A_norm = d_inv_sqrt.unsqueeze(1) * A_hat * d_inv_sqrt.unsqueeze(0)

    # Train/val/test split — 20% train, 20% val, 60% test (per class)
    train_mask = torch.zeros(N, dtype=torch.bool, device=device)
    val_mask = torch.zeros(N, dtype=torch.bool, device=device)
    rng = np.random.default_rng(0)
    for c in range(n_classes):
        idx = np.where(y_np == c)[0]
        if len(idx) == 0:
            continue
        rng.shuffle(idx)
        n_train = max(1, int(0.2 * len(idx)))
        n_val = max(1, int(0.2 * len(idx)))
        train_mask[idx[:n_train]] = True
        val_mask[idx[n_train : n_train + n_val]] = True
    test_mask = ~(train_mask | val_mask)
    print(
        f"  train: {int(train_mask.sum().item())}, "
        f"val: {int(val_mask.sum().item())}, "
        f"test: {int(test_mask.sum().item())}"
    )

    return {
        "X": X,
        "A": A,
        "y": y,
        "A_norm": A_norm,
        "A_hat": A_hat,
        "X_np": X_np,
        "A_np": A_np,
        "y_np": y_np,
        "edge_index_np": edge_index_np,
        "train_mask": train_mask,
        "val_mask": val_mask,
        "test_mask": test_mask,
        "N": N,
        "F_dim": F_dim,
        "n_classes": n_classes,
        "n_edges_undirected": n_edges_undirected,
        "dataset_name": dataset_name,
    }


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_gnns.db"
    registry_db = "sqlite:///mlfp05_gnns_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_gnns", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# TRAINING HARNESS
# ════════════════════════════════════════════════════════════════════════


def train_node_classifier(
    model: nn.Module,
    name: str,
    forward_arg: torch.Tensor,
    graph_data: dict,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = 100,
    lr: float = 1e-2,
    weight_decay: float = 5e-4,
) -> tuple[list[float], list[float], list[float]]:
    """Train a GNN for node classification and log metrics to ExperimentTracker.

    Returns:
        train_losses: per-epoch training loss
        val_accs: per-epoch validation accuracy
        test_accs: per-epoch test accuracy
    """
    return asyncio.run(
        _train_node_classifier_async(
            model,
            name,
            forward_arg,
            graph_data,
            tracker,
            exp_name,
            epochs,
            lr,
            weight_decay,
        )
    )


async def _train_node_classifier_async(
    model: nn.Module,
    name: str,
    forward_arg: torch.Tensor,
    graph_data: dict,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = 100,
    lr: float = 1e-2,
    weight_decay: float = 5e-4,
) -> tuple[list[float], list[float], list[float]]:
    """Async core — uses the kailash-ml 1.1.1 ``tracker.track(...)`` context manager."""
    X = graph_data["X"]
    y = graph_data["y"]
    train_mask = graph_data["train_mask"]
    val_mask = graph_data["val_mask"]
    test_mask = graph_data["test_mask"]
    N = graph_data["N"]
    n_edges = graph_data["n_edges_undirected"]
    dataset_name = graph_data["dataset_name"]
    hidden_dim = 16 if dataset_name == "Karate Club" else 64

    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  [{name}] parameters: {n_params:,}")

    train_losses: list[float] = []
    val_accs: list[float] = []
    test_accs: list[float] = []

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "hidden_dim": str(hidden_dim),
                "epochs": str(epochs),
                "lr": str(lr),
                "weight_decay": str(weight_decay),
                "n_params": str(n_params),
                "dataset": dataset_name,
                "n_nodes": str(N),
                "n_edges": str(n_edges),
            }
        )

        for epoch in range(epochs):
            model.train()
            opt.zero_grad()
            logits = model(X, forward_arg)
            loss = F.cross_entropy(logits[train_mask], y[train_mask])
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

            model.eval()
            with torch.no_grad():
                preds = model(X, forward_arg).argmax(dim=-1)
                v_acc = (preds[val_mask] == y[val_mask]).float().mean().item()
                t_acc = (preds[test_mask] == y[test_mask]).float().mean().item()
            val_accs.append(v_acc)
            test_accs.append(t_acc)

            await run.log_metrics(
                {
                    "train_loss": loss.item(),
                    "val_accuracy": v_acc,
                    "test_accuracy": t_acc,
                },
                step=epoch + 1,
            )

            if (epoch + 1) % 25 == 0:
                print(
                    f"  [{name}] epoch {epoch+1:3d}  "
                    f"loss={loss.item():.4f}  val_acc={v_acc:.3f}  test_acc={t_acc:.3f}"
                )

        await run.log_metrics(
            {
                "final_loss": train_losses[-1],
                "final_val_accuracy": val_accs[-1],
                "final_test_accuracy": test_accs[-1],
                "best_val_accuracy": max(val_accs),
                "best_test_accuracy": max(test_accs),
            }
        )

    return train_losses, val_accs, test_accs


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION HELPERS
# ════════════════════════════════════════════════════════════════════════

viz = ModelVisualizer()


def plot_training_curves(
    metrics_dict: dict[str, list[float]],
    title: str,
    y_label: str,
    filename: str,
) -> None:
    """Plot overlaid training curves and save as HTML."""
    fig = viz.training_history(
        metrics=metrics_dict,
        x_label="Epoch",
        y_label=y_label,
    )
    fig.update_layout(title=title)
    filepath = OUTPUT_DIR / filename
    fig.write_html(str(filepath))
    print(f"  Saved: {filepath}")


def plot_node_embeddings(
    embeddings: np.ndarray,
    labels: np.ndarray,
    n_classes: int,
    title: str,
    filename: str,
) -> None:
    """2-D PCA projection of node embeddings coloured by class label.

    Uses SVD-based PCA (no sklearn dependency). Nodes of the same class
    should cluster together if the GNN learned meaningful representations.
    """
    emb_centered = embeddings - embeddings.mean(axis=0, keepdims=True)
    Vt = np.linalg.svd(emb_centered, full_matrices=False)[2]
    coords = emb_centered @ Vt.T[:, :2]

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    cmap = plt.cm.get_cmap("tab10", n_classes)
    for c in range(n_classes):
        mask = labels == c
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=[cmap(c)],
            s=15,
            alpha=0.6,
            label=f"Class {c}",
        )
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    ax.legend(fontsize=8, markerscale=2, loc="best")
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")

    # Text summary of first 3 nodes per class
    print(f"\n  {title} — first 3 nodes per class:")
    for c in range(min(n_classes, 7)):
        rows = coords[labels == c][:3]
        if len(rows) == 0:
            continue
        pretty = ", ".join(f"({r[0]:+.2f}, {r[1]:+.2f})" for r in rows)
        print(f"    class {c}: {pretty}")

    return coords


def plot_graph_with_embeddings(
    A_np: np.ndarray,
    embeddings_2d: np.ndarray,
    labels: np.ndarray,
    n_classes: int,
    title: str,
    filename: str,
    max_nodes: int = 200,
) -> None:
    """Draw graph edges on the 2-D embedding space, coloured by class.

    Subsamples to max_nodes for readability on large graphs.
    """
    N = A_np.shape[0]
    if N > max_nodes:
        rng = np.random.default_rng(42)
        subset = rng.choice(N, max_nodes, replace=False)
    else:
        subset = np.arange(N)

    coords = embeddings_2d[subset]
    sub_labels = labels[subset]
    sub_A = A_np[np.ix_(subset, subset)]

    fig, ax = plt.subplots(1, 1, figsize=(12, 9))
    cmap = plt.cm.get_cmap("tab10", n_classes)

    # Draw edges first (behind nodes)
    src_idx, dst_idx = np.where(np.triu(sub_A) > 0)
    for s, d in zip(src_idx, dst_idx):
        ax.plot(
            [coords[s, 0], coords[d, 0]],
            [coords[s, 1], coords[d, 1]],
            color="gray",
            alpha=0.08,
            linewidth=0.5,
        )

    # Draw nodes
    for c in range(n_classes):
        mask = sub_labels == c
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=[cmap(c)],
            s=25,
            alpha=0.7,
            label=f"Class {c}",
            zorder=2,
        )

    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.legend(fontsize=8, markerscale=2, loc="best")
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")


def plot_attention_weights(
    alpha: np.ndarray,
    A_np: np.ndarray,
    labels: np.ndarray,
    title: str,
    filename: str,
    node_idx: int = 0,
    top_k: int = 10,
) -> None:
    """Visualise attention weights for a single node's neighbourhood.

    Shows which neighbours the GAT layer attends to most strongly.
    """
    neighbours = np.where(A_np[node_idx] > 0)[0]
    if len(neighbours) == 0:
        print(f"  Node {node_idx} has no neighbours — skipping attention plot")
        return

    weights = alpha[node_idx, neighbours]
    order = np.argsort(-weights)[:top_k]
    top_neighbours = neighbours[order]
    top_weights = weights[order]

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    bar_colors = [plt.cm.get_cmap("tab10")(labels[n] % 10) for n in top_neighbours]
    ax.barh(
        range(len(top_neighbours)),
        top_weights,
        color=bar_colors,
        edgecolor="white",
    )
    ax.set_yticks(range(len(top_neighbours)))
    ax.set_yticklabels(
        [f"Node {n} (class {labels[n]})" for n in top_neighbours],
        fontsize=9,
    )
    ax.set_xlabel("Attention Weight")
    ax.set_title(
        f"{title}\nNode {node_idx} (class {labels[node_idx]}) attending to top-{top_k} neighbours",
        fontsize=12,
        fontweight="bold",
    )
    ax.invert_yaxis()
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION HELPER
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics: list[MetricSpec],
) -> object:
    """Register a model in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=metrics,
    )
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics: list[MetricSpec],
) -> object:
    """Synchronously register a model."""
    return asyncio.run(_register_model(registry, name, model, metrics))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 6.4: Link Prediction with GNN Encoder-Decoder
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Why link prediction matters (knowledge graphs, social networks, recs)
#   - Encoder-decoder architecture: GNN encoder + dot-product decoder
#   - Positive vs negative edge sampling for training
#   - AUC metric for ranking quality evaluation
#   - Train a link predictor on the Cora citation network
#   - Track training with kailash-ml ExperimentTracker
#
# PREREQUISITES: M5/ex_6.1 (GCN layer implementation).
# ESTIMATED TIME: ~30 min
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from kailash_ml.types import MetricSpec

import matplotlib.pyplot as plt



## PHASE 1 — THEORY: Why Link Prediction Matters


LINK PREDICTION: "Should these two nodes be connected?"

  THREE MAJOR APPLICATIONS:
  1. Knowledge Graphs: discover new drug-disease interactions
  2. Social Networks: "people you may know" suggestions
  3. Citation Networks: "papers you should cite" recommendations

  ENCODER-DECODER APPROACH:
  - ENCODER (GNN): learns node embeddings z_i from features + structure
  - DECODER (dot product): score(i,j) = sigmoid(z_i^T z_j)
  - High similarity in embedding space -> predict edge exists

  TRAINING DATA:
  - Positive samples: real edges from the graph (label = 1)
  - Negative samples: random non-edges (label = 0)
  - Loss: binary cross-entropy on edge predictions
  - Metric: AUC — how well do we rank real edges above non-edges?


In [ ]:
#
# Node classification asks "what is this node?" Link prediction asks
# "should these two nodes be connected?" This is the foundation of:
#
# 1. KNOWLEDGE GRAPHS: A medical knowledge graph has nodes for drugs,
#    diseases, proteins, and genes. Edges represent known interactions
#    (drug X treats disease Y). Link prediction discovers MISSING edges
#    — potential new drug-disease interactions that haven't been tested.
#
# 2. SOCIAL NETWORKS: "People you may know" = predict missing friendship
#    edges based on mutual connections and profile features.
#
# 3. RECOMMENDATION: "Papers you should cite" = predict missing citation
#    edges based on content similarity and citation patterns.
#
# The approach: ENCODER-DECODER architecture
#   - ENCODER: A GNN (like our GCN) that produces node embeddings z_i
#   - DECODER: dot-product similarity between embeddings
#     score(i, j) = sigmoid( z_i^T z_j )
#
# If two nodes have similar embeddings, the decoder predicts an edge
# between them. Training uses known edges as positives and random
# non-edges as negatives — binary classification on edge existence.
print("=" * 70)
print("  PHASE 1 — THEORY: Link Prediction on Graphs")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: GNN Encoder + Dot-Product Decoder


GCN layer: H' = A_norm @ (H @ W).


Encoder: MLP -> two GCN layers that produce node embeddings.
    Decoder: dot product between node pairs -> edge probability.

    The encoder first projects features through an MLP (to handle
    high-dimensional bag-of-words features), then applies two GCN
    layers to incorporate graph structure into the embeddings.


Produce node embeddings: MLP -> GCN -> GCN.


Dot-product decoder: score(i,j) = z_i^T z_j.


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: Link Prediction Model")
print("=" * 70)

# Load graph data and set up engines
graph_data = load_graph_data()
conn, tracker, exp_name, registry, has_registry = setup_engines()

X = graph_data["X"]
A = graph_data["A"]
A_norm = graph_data["A_norm"]
A_np = graph_data["A_np"]
y_np = graph_data["y_np"]
edge_index_np = graph_data["edge_index_np"]
N = graph_data["N"]
F_dim = graph_data["F_dim"]
n_classes = graph_data["n_classes"]
dataset_name = graph_data["dataset_name"]

HIDDEN_DIM = 16 if dataset_name == "Karate Club" else 64
LINK_EPOCHS = 80


# Reuse GCN layer for the encoder
class GCNLayer(nn.Module):

    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=True)

    def forward(self, h: torch.Tensor, a_norm: torch.Tensor) -> torch.Tensor:
        return a_norm @ self.W(h)


class LinkPredictor(nn.Module):

    def __init__(self, in_dim: int, hidden_dim: int):
        super().__init__()
        # TODO: Build the encoder — an MLP followed by two GCN layers:
        # 1. self.encoder = nn.Sequential(
        #        nn.Linear(in_dim, hidden_dim),
        #        nn.ReLU(),
        #        nn.Linear(hidden_dim, hidden_dim),
        #    )
        # 2. self.gcn1 = GCNLayer(hidden_dim, hidden_dim)
        # 3. self.gcn2 = GCNLayer(hidden_dim, hidden_dim)
        pass

    def encode(self, h: torch.Tensor, a_norm: torch.Tensor) -> torch.Tensor:
        # TODO: Pass h through encoder MLP, then GCN1 with ReLU, then GCN2
        # h = self.encoder(h)
        # h = F.relu(self.gcn1(h, a_norm))
        # h = self.gcn2(h, a_norm)
        # return h
        pass

    def decode(
        self, z: torch.Tensor, src: torch.Tensor, dst: torch.Tensor
    ) -> torch.Tensor:
        # TODO: Compute dot product between source and destination embeddings
        # Hint: return (z[src] * z[dst]).sum(dim=-1)
        pass

    def forward(
        self,
        h: torch.Tensor,
        a_norm: torch.Tensor,
        src: torch.Tensor,
        dst: torch.Tensor,
    ) -> torch.Tensor:
        z = self.encode(h, a_norm)
        return self.decode(z, src, dst)


link_model = LinkPredictor(F_dim, HIDDEN_DIM).to(device)
n_params = sum(p.numel() for p in link_model.parameters())

print(f"\n  Link Prediction architecture:")
print(
    f"    Encoder MLP: Linear({F_dim}->{HIDDEN_DIM}) -> ReLU -> Linear({HIDDEN_DIM}->{HIDDEN_DIM})"
)
print(f"    GCN Layer 1: GCNLayer({HIDDEN_DIM} -> {HIDDEN_DIM})")
print(f"    GCN Layer 2: GCNLayer({HIDDEN_DIM} -> {HIDDEN_DIM})")
print(f"    Decoder: dot_product(z_i, z_j) -> edge score")
print(f"    Total parameters: {n_params:,}")



### Build Checkpoint


In [ ]:
assert isinstance(link_model, nn.Module), "LinkPredictor should be an nn.Module"
assert n_params > 0, "LinkPredictor should have learnable parameters"
print("\n--- Build checkpoint passed --- LinkPredictor architecture created\n")



## PHASE 3 — TRAIN: Link Prediction on Cora


Train the link predictor under a tracker.track(...) context.


In [ ]:
print("=" * 70)
print(f"  PHASE 3 — TRAIN: Link Prediction on {dataset_name}")
print("=" * 70)

# Prepare positive and negative edge samples
# Positive: real edges from the graph
pos_src = torch.from_numpy(edge_index_np[0]).to(device)
pos_dst = torch.from_numpy(edge_index_np[1]).to(device)
n_pos = len(pos_src)

# TODO: Sample negative edges — random node pairs that are NOT connected
# 1. Use rng = np.random.default_rng(42)
# 2. Repeatedly sample random (s, d) pairs where s != d and A_np[s, d] == 0
# 3. Collect n_pos negative edges to match the positive count
# 4. Convert to tensors on device
# Hint: while neg_count < n_pos:
#          s = rng_link.integers(0, N); d = rng_link.integers(0, N)
#          if s != d and A_np[s, d] == 0: add to lists
neg_src_list = []
neg_dst_list = []
rng_link = np.random.default_rng(42)
# TODO: Fill neg_src_list and neg_dst_list
neg_count = 0
# while neg_count < n_pos:
#     s = rng_link.integers(0, N)
#     d = rng_link.integers(0, N)
#     if s != d and A_np[s, d] == 0:
#         neg_src_list.append(s)
#         neg_dst_list.append(d)
#         neg_count += 1
neg_src = torch.tensor(neg_src_list, dtype=torch.long, device=device)
neg_dst = torch.tensor(neg_dst_list, dtype=torch.long, device=device)

print(f"  Positive edges (real connections): {n_pos:,}")
print(f"  Negative edges (random non-edges): {len(neg_src):,}")
print(f"  Ratio: 1:1 (balanced)")

# Train
link_opt = torch.optim.Adam(link_model.parameters(), lr=1e-2, weight_decay=1e-4)
link_losses: list[float] = []
link_aucs: list[float] = []


async def _train_link_predictor_async():
    async with tracker.track(experiment=exp_name, run_name="link_prediction") as run:
        await run.log_params(
            {
                "task": "link_prediction",
                "hidden_dim": str(HIDDEN_DIM),
                "epochs": str(LINK_EPOCHS),
                "n_pos_edges": str(n_pos),
                "n_neg_edges": str(len(neg_src)),
            }
        )

        for epoch in range(LINK_EPOCHS):
            link_model.train()
            link_opt.zero_grad()

            # TODO: Compute positive and negative scores, then BCE loss
            # 1. pos_scores = link_model(X, A_norm, pos_src, pos_dst)
            # 2. neg_scores = link_model(X, A_norm, neg_src, neg_dst)
            # 3. Concatenate scores and labels:
            #    scores = torch.cat([pos_scores, neg_scores])
            #    labels = torch.cat([ones for pos, zeros for neg])
            # 4. loss = F.binary_cross_entropy_with_logits(scores, labels)
            # 5. loss.backward(); link_opt.step()
            # Hint: torch.ones(n_pos, device=device) for positive labels

            # TODO: Compute AUC-like metric
            # link_model.eval()
            # with torch.no_grad():
            #     p_scores = link_model(X, A_norm, pos_src, pos_dst)
            #     n_scores = link_model(X, A_norm, neg_src, neg_dst)
            #     n_sample = min(1000, n_pos, len(neg_src))
            #     auc_approx = (p_scores[:n_sample] > n_scores[:n_sample]).float().mean().item()
            # link_aucs.append(auc_approx)

            # TODO: Log metrics
            # await run.log_metrics({"link_loss": loss.item(), "link_auc_approx": auc_approx}, step=epoch+1)

            if (epoch + 1) % 20 == 0:
                print(
                    f"  [LinkPred] epoch {epoch+1:3d}  "
                    f"loss={link_losses[-1] if link_losses else 0:.4f}  "
                    f"auc_approx={link_aucs[-1] if link_aucs else 0:.3f}"
                )

        if link_losses and link_aucs:
            await run.log_metrics(
                {
                    "final_link_loss": link_losses[-1],
                    "final_link_auc": link_aucs[-1],
                }
            )


await _train_link_predictor_async()



### Train Checkpoint


In [ ]:
assert len(link_losses) == LINK_EPOCHS, "Link prediction should train for all epochs"
assert link_losses[-1] < link_losses[0], "Link prediction loss should decrease"
assert (
    link_aucs[-1] > 0.55
), f"Link prediction AUC {link_aucs[-1]:.3f} should exceed random (0.5)"
final_auc = link_aucs[-1]
print(f"\n  Link Prediction Results:")
print(f"    Final loss:      {link_losses[-1]:.4f}")
print(f"    Final AUC:       {final_auc:.4f}")
print(f"    Best AUC:        {max(link_aucs):.4f}")
print(f"    Random baseline: 0.5000")
# INTERPRETATION: The link predictor learns that connected papers have
# similar GNN embeddings. The dot-product decoder measures embedding
# similarity — high similarity predicts a citation link. An AUC > 0.5
# means the model ranks real edges higher than random non-edges.
print("\n--- Train checkpoint passed --- link prediction trained\n")



## PHASE 4 — VISUALISE: Edge Scores + Embedding Similarity


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: Link Prediction Analysis")
print("=" * 70)

link_model.eval()
with torch.no_grad():
    # Get node embeddings
    z = link_model.encode(X, A_norm).cpu().numpy()

    # Score distributions for positive vs negative edges
    pos_final_scores = link_model(X, A_norm, pos_src, pos_dst).cpu().numpy()
    neg_final_scores = link_model(X, A_norm, neg_src, neg_dst).cpu().numpy()

# TODO: Create visualisation with 2 subplots:
# Left: Histogram of score distributions (positive vs negative edges)
#   - pos_final_scores[:2000] in green with label "Real edges"
#   - neg_final_scores[:2000] in red with label "Non-edges"
#   - density=True, bins=60
# Right: Training progress (loss and AUC on twin y-axes)
#   - Loss on left y-axis in steelblue
#   - AUC on right y-axis in coral (use ax.twinx())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TODO: Fill in the histogram and training curve plots
# axes[0].hist(pos_final_scores[:2000], bins=60, alpha=0.7, color="green", ...)
# axes[0].hist(neg_final_scores[:2000], bins=60, alpha=0.7, color="red", ...)

fig.tight_layout()
filepath = OUTPUT_DIR / "link_prediction_analysis.png"
fig.savefig(filepath, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {filepath}")

# TODO: Create heatmap comparing true adjacency vs predicted similarity
# 1. Select n_sub=50 random nodes
# 2. Compute dot-product similarity: z_sub @ z_sub.T
# 3. Plot side-by-side: true adjacency matrix vs similarity matrix
# Hint: fig, axes = plt.subplots(1, 2, figsize=(14, 6))
#        axes[0].imshow(sub_A, cmap="Blues")
#        axes[1].imshow(similarity, cmap="RdBu_r")
n_sub = min(50, N)
rng = np.random.default_rng(42)
sub_idx = rng.choice(N, n_sub, replace=False)
sub_idx = np.sort(sub_idx)
z_sub = z[sub_idx]
similarity = z_sub @ z_sub.T  # dot-product similarity
sub_A = A_np[np.ix_(sub_idx, sub_idx)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# TODO: Plot true adjacency and predicted similarity side-by-side

fig.suptitle(
    f"Link Prediction: True Edges vs Learned Similarities — {dataset_name}",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
filepath = OUTPUT_DIR / "link_prediction_similarity.png"
fig.savefig(filepath, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {filepath}")

# Training curves via ModelVisualizer
plot_training_curves(
    metrics_dict={
        "Link pred loss": link_losses,
        "Link pred AUC (approx)": link_aucs,
    },
    title="Link Prediction Training",
    y_label="Value",
    filename="link_prediction_curves.html",
)

# Score statistics
pos_mean = pos_final_scores.mean()
neg_mean = neg_final_scores.mean()
print(f"\n  Score analysis:")
print(f"    Real edges   — mean score: {pos_mean:+.4f}")
print(f"    Non-edges    — mean score: {neg_mean:+.4f}")
print(f"    Separation:  {pos_mean - neg_mean:.4f}")
print(f"    -> Good separation means the model can distinguish real from fake edges")



### Visualise Checkpoint


In [ ]:
assert z.shape == (N, HIDDEN_DIM), f"Embedding shape should be ({N}, {HIDDEN_DIM})"
assert pos_mean > neg_mean, "Positive edges should score higher on average"
print("\n--- Visualise checkpoint passed --- link prediction analysis plotted\n")



## PHASE 5 — APPLY: Knowledge Graph Completion for a Hospital


SCENARIO: You're building a drug-disease interaction predictor for
  Singapore General Hospital (SGH) using a medical knowledge graph.

  THE KNOWLEDGE GRAPH:
  - Drug nodes: ~5K approved drugs (features: molecular weight, targets, ATC code)
  - Disease nodes: ~10K conditions (features: ICD-10 code, organ system, prevalence)
  - Protein nodes: ~20K proteins (features: function, pathway, expression level)
  - Edges: known interactions (drug-treats-disease, drug-binds-protein,
    protein-associated-with-disease)

  LINK PREDICTION TASK: Discover new drug-disease edges
  - Known: drug X treats disease Y (from clinical trials)
  - Unknown: does drug X also treat disease Z? (drug repurposing)
  - Validation: withhold 20% of known edges, predict them

  HOW IT WORKS:
  1. Encode all nodes with GNN: each drug gets an embedding that
     captures its molecular features AND its known interactions
  2. Score all (drug, disease) pairs with dot product
  3. Rank by score — top-k are candidate interactions for lab testing
  4. Validate: do withheld known interactions appear in top-k?


CLINICAL DEPLOYMENT:
  1. Build KG from DrugBank, OMIM, STRING databases + SGH clinical records
  2. Train link predictor on known drug-disease edges
  3. Score all (drug, disease) pairs without known interactions
  4. Top-k candidates reviewed by pharmacology team for literature evidence
  5. Promising candidates enter pre-clinical or retrospective cohort studies
  6. Track predictions with ExperimentTracker — validate against new trial results
  7. Register model version in ModelRegistry with AUC and recall@k metrics


In [ ]:
print("=" * 70)
print("  PHASE 5 — APPLY: Knowledge Graph Completion at SGH")
print("=" * 70)
print(
)

# TODO: Demonstrate edge rediscovery experiment
# 1. Withhold 100 real edges (randomly selected)
# 2. Score withheld edges with the trained model
# 3. Score random non-edges for comparison
# 4. Compute rediscovery AUC: fraction of withheld edges scoring above random
# Hint: withheld_scores = link_model(X, A_norm, withheld_src, withheld_dst)
#        random_scores = link_model(X, A_norm, rand_src, rand_dst)
#        rediscovery_auc = (withheld_scores > random_scores).float().mean().item()
print("  Demonstration: edge rediscovery experiment")
print("  (Withholding 100 real edges, checking if the model scores them highly)\n")

rng_demo = np.random.default_rng(123)
n_test_edges = min(100, n_pos)
test_edge_indices = rng_demo.choice(n_pos, n_test_edges, replace=False)

withheld_src = pos_src[test_edge_indices]
withheld_dst = pos_dst[test_edge_indices]

# TODO: Score withheld edges and compare to random non-edges
# with torch.no_grad():
#     withheld_scores = link_model(X, A_norm, withheld_src, withheld_dst)
#     rand_src = torch.randint(0, N, (n_test_edges,), device=device)
#     rand_dst = torch.randint(0, N, (n_test_edges,), device=device)
#     random_scores = link_model(X, A_norm, rand_src, rand_dst)
#
# withheld_mean = withheld_scores.mean().item()
# random_mean = random_scores.mean().item()
# rediscovery_auc = (withheld_scores > random_scores).float().mean().item()

withheld_mean = 0.0  # Replace with your computed value
random_mean = 0.0  # Replace with your computed value
rediscovery_auc = 0.0  # Replace with your computed value

print(f"    Withheld real edges — mean score: {withheld_mean:+.4f}")
print(f"    Random non-edges   — mean score: {random_mean:+.4f}")
print(f"    Rediscovery AUC:                  {rediscovery_auc:.4f}")
print(f"    (1.0 = perfectly ranks real edges above non-edges)")

if rediscovery_auc > 0.6:
    print("    -> Model successfully rediscovers withheld edges!")
    print(
        "    -> In a hospital setting: these would be drug-disease candidates for trials"
    )

print(
)

# Register the link predictor
if has_registry:
    version = register_model(
        registry=registry,
        name="m5_gnn_link_predictor",
        model=link_model,
        metrics=[
            MetricSpec(name="final_link_auc", value=final_auc),
            MetricSpec(name="final_link_loss", value=link_losses[-1]),
            MetricSpec(name="rediscovery_auc", value=rediscovery_auc),
        ],
    )
    print(
        f"  Registered link_predictor: version={version.version}, auc={final_auc:.4f}"
    )



### Apply Checkpoint


In [ ]:
assert rediscovery_auc > 0.5, "Rediscovery AUC should beat random"
print("\n--- Apply checkpoint passed --- knowledge graph completion demonstrated\n")



## REFLECTION


LINK PREDICTION WITH GNN ENCODER-DECODER:
  [x] Encoder: GCN layers produce node embeddings from features + structure
  [x] Decoder: dot-product similarity — score(i,j) = z_i^T z_j
  [x] Training: positive edges (real) vs negative edges (sampled non-edges)
  [x] AUC metric: {final_auc:.1%} — ranks real edges above non-edges
  [x] Rediscovery experiment: {rediscovery_auc:.1%} AUC on withheld edges
  [x] Visualised score distributions and similarity heatmaps

  LINK PREDICTION vs NODE CLASSIFICATION:
  - Node classification: "what IS this node?" (label prediction)
  - Link prediction: "should these nodes be CONNECTED?" (edge prediction)
  - Same GNN encoder; different decoder (classifier vs dot product)
  - Link prediction is the foundation of recommendation systems

  APPLICATIONS:
  - Knowledge graphs: discover new drug-disease interactions
  - Social networks: "people you may know"
  - Citation networks: "papers you should cite"
  - E-commerce: "products frequently bought together"

  Next: Exercise 6.5 — Architecture Comparison: systematic side-by-side
  evaluation of GCN vs GAT vs GraphSAGE on the same dataset...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — Link Prediction")
print("=" * 70)
print(
)

# Clean up
await conn.close()



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# Reference: `kailash_ml.diagnostics` (via `kailash-ml`) — see gold standard
# `solutions/ex_1/01_standard_ae.py` for the full pattern.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    # Link prediction BCE over node pair dot products
    # Customise per your exercise's loss shape.
    if isinstance(batch, (tuple, list)):
        x = batch[0]
        y = batch[1] if len(batch) > 1 else None
    else:
        x, y = batch, None
    out = m(x)
    import torch.nn.functional as F
    if y is None:
        return F.mse_loss(out, x)
    return F.cross_entropy(out, y)


print("\n── Diagnostic Report (Link Prediction with GNN) ──")
try:
    diag, findings = run_diagnostic_checkpoint(
        link_model,
        edge_loader,
        _diag_loss,
        title="Link Prediction with GNN",
        n_batches=8,
        show=False,
    )
except Exception as exc:
    # Diagnostic is pedagogical — never block the exercise on it.
    print(f"[diagnostic skipped: {exc}]")

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
# [✓] Gradient flow (HEALTHY): RMS 5.1e-04 to 7.3e-03.
# [!] Loss trend    (WARNING): train loss → 0.12 but val AUC plateaus at 0.89.
#     Signature of train-val gap — model memorising specific edges.



## STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:


In [ ]:
#  [STETHOSCOPE] Link prediction overfits fast because the task
#     is effectively "memorise these specific edges". The val AUC
#     plateau while train loss keeps dropping is the canonical
#     overfit signature slide 5.3's Stethoscope teaches.
#     >> Prescription: add negative sampling diversity, use dropout
#        on edges (DropEdge), or reduce embedding dimensionality.
#
#  [BLOOD TEST] Healthy gradients. The issue is data-side (limited
#     positive edges), not optimisation-side.

